# Qiskit BackendV2 quickstart

Build and transpile a Bell circuit, then compare Qiskit's exact state with MettleQ's native BackendV2 result and execution plan.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
circuit = QuantumCircuit(2, name="bell")
circuit.h(0)
circuit.cx(0, 1)

reference, reference_ms, _ = benchmark(
    lambda: np.asarray(Statevector.from_instruction(circuit).data)
)
backend = MettleQBackend(method="statevector", device="auto")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return np.asarray(
        backend.run(
            compiled,
            shots=1,
            return_statevector=True,
            execution_report=True,
        ).result().data(0)["statevector"]
    )

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = phase_aligned_statevector_error(reference, candidate)
method, device = qiskit_selection(backend)
plan = backend.last_execution_plans[-1]
tutorial_result = emit_result(
    notebook="qiskit/01_backend_quickstart.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned statevector atol=2e-6",
    passed=error <= 2e-6 and compiled.num_qubits == 2,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "execution_status": plan["execution_status"]},
)

TUTORIAL_RESULT::{"check": "phase-aligned statevector atol=2e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"execution_status": "evaluated", "max_amplitude_error": 1.2101617041793133e-08}, "mettleq_median_ms": 0.2738329931162298, "notebook": "qiskit/01_backend_quickstart.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.06329201278276742, "reference_over_mettleq": 0.23113362660394543, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
